In [12]:
from sdv.single_table import GaussianCopulaSynthesizer
from sdv.metadata import SingleTableMetadata
import pandas as pd
import numpy as np




In [13]:
def generate_synthetic_with_lags(df, lag_max=6, n_samples=20, random_state=42):
    np.random.seed(random_state)

    df = df.copy()
    df["QuarterEnd"] = pd.to_datetime(df["QuarterEnd"])
    df = df.sort_values("QuarterEnd").reset_index(drop=True)

    # creating lagged features
    feature_cols = [c for c in df.columns if c not in ["QuarterEnd", "Year"]]

    for col in feature_cols:
        for lag in range(1, lag_max + 1):
            df[f"{col}_lag{lag}"] = df[col].shift(lag)

    df = df.dropna().reset_index(drop=True)

    # metadata (mandatory)
    metadata = SingleTableMetadata()
    metadata.detect_from_dataframe(df.drop(columns=["QuarterEnd"]))

    # fit copula
    copula = GaussianCopulaSynthesizer(metadata)
    copula.fit(df.drop(columns=["QuarterEnd"]))

    # sample
    synth = copula.sample(n_samples)

    # synthetica dates
    first_date = df["QuarterEnd"].min()
    synthetic_dates = pd.date_range(end=first_date, periods=n_samples+1, freq="Q")[:-1]

    synth["QuarterEnd"] = synthetic_dates
    synth = synth.sort_values("QuarterEnd").reset_index(drop=True)

    # merging
    final = pd.concat([synth, df], ignore_index=True)
    final = final.sort_values("QuarterEnd").reset_index(drop=True)

    return final


In [14]:
df = pd.read_csv("master_dataset_modelling.csv")

df_complete = generate_synthetic_with_lags(df, lag_max=12, n_samples=200, random_state=42)


/var/folders/z_/h8pm6bx50mngk0pbk5p07btc0000gn/T/ipykernel_1578/1435476766.py:13: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

/var/folders/z_/h8pm6bx50mngk0pbk5p07btc0000gn/T/ipykernel_1578/1435476766.py:13: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

/var/folders/z_/h8pm6bx50mngk0pbk5p07btc0000gn/T/ipykernel_1578/1435476766.py:13: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.co

In [15]:
lagged_cols = [col for col in df_complete.columns if any(col.endswith(f'_lag{i}') for i in range(1, 13))]

# Removes lagged features created previously
df_complete.drop(columns=lagged_cols, inplace=True)


In [16]:
df_complete.to_csv("synthetic_dataset.csv")